In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
import numpy as np
import pandas as pd

# -----------------------------
# SETTINGS
# -----------------------------
radius = 2
nBits = 2048
k = 5

# -----------------------------
# LOAD TRAINING DATA
# -----------------------------
train_df = pd.read_csv("dataset/chembl_data/final_standard_smiles.csv")
hit_df = pd.read_csv("result/alt_ZINC_ChEMBL_RF_Morgan_high_confidence.csv")
zinc_df = pd.read_csv("dataset/zinc/standardized_zinc_with_ids.csv")
hit_df = hit_df.merge(zinc_df, on="ZINC_ID", how="left")

# -----------------------------
# FINGERPRINT FUNCTION
# -----------------------------
def mol_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=nBits
    )

# -----------------------------
# TRAINING FINGERPRINTS
# -----------------------------
train_fps = [
    mol_to_fp(sm)
    for sm in train_df["standard_smiles"]
]

# -----------------------------
# COMPUTE TRAINING DISTANCES
# -----------------------------
training_knn_distances = []

for i, fp in enumerate(train_fps):

    sims = []

    for j, other_fp in enumerate(train_fps):

        if i != j:
            sim = DataStructs.TanimotoSimilarity(fp, other_fp)
            sims.append(sim)

    sims = sorted(sims, reverse=True)

    knn_sims = sims[:k]

    avg_distance = np.mean([1 - s for s in knn_sims])

    training_knn_distances.append(avg_distance)

# -----------------------------
# DEFINE THRESHOLD
# -----------------------------
threshold = np.mean(training_knn_distances) + 2 * np.std(training_knn_distances)

print("Applicability Domain Threshold:", threshold)

# -----------------------------
# EVALUATE PRIORITIZED COMPOUNDS
# -----------------------------
results = []

for _, row in hit_df.iterrows():

    sm = row["standard_smiles"]

    fp = mol_to_fp(sm)

    sims = [
        DataStructs.TanimotoSimilarity(fp, train_fp)
        for train_fp in train_fps
    ]

    sims = sorted(sims, reverse=True)

    knn_sims = sims[:k]

    avg_distance = np.mean([1 - s for s in knn_sims])

    inside_ad = avg_distance <= threshold

    results.append({
        "ZINC_ID": row["ZINC_ID"],
        "standard_smiles": sm,
        "Average_kNN_Distance": avg_distance,
        "Inside_AD": inside_ad,
        "AD_Threshold": threshold
    })

results_df = pd.DataFrame(results)

results_df.to_csv("result/zinc_hit_AD.csv", index=False)
print(results_df)